In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
import mlflow
import mlflow.sklearn

np.random.seed(42)

In [2]:
df_train = pd.read_csv("train_set.csv")
df_test = pd.read_csv("test_set.csv")
df_train.shape, df_test.shape

((315, 3240), (100, 3240))

In [3]:
X_train_raw = df_train.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_train = df_train['CLASS']
X_test_raw = df_test.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_test = df_test['CLASS']

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

In [6]:
# ml flow setup 
mlflow.set_tracking_uri("file:./mlruns2")
experiment_name = "Classification_Experiment"

if not mlflow.get_experiment_by_name(experiment_name):
	mlflow.create_experiment(experiment_name)

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:///d:/After/Projects/MEDICAL_IMAGING/NAAMII_ENTRANCE/Medical-Imaging-Tasks/MachineLearningTask/mlruns2/306129573321834715', creation_time=1748185548721, experiment_id='306129573321834715', last_update_time=1748185548721, lifecycle_stage='active', name='Classification_Experiment', tags={}>

In [7]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, log_loss
import matplotlib.pyplot as plt
import sklearn 
import seaborn as sns
import os 

In [ ]:
def evaluate_model(model, X_test_data, y_test_data, model_name, run_id):
    preds = model.predict(X_test_data)
    proba_preds = model.predict_proba(X_test_data)[:, 1]

    accuracy = accuracy_score(y_test_data, preds)
    logloss = log_loss(y_test_data, proba_preds)
    fpr, tpr, _ = roc_curve(y_test_data, proba_preds)
    roc_auc = auc(fpr, tpr)

    with mlflow.start_run(run_id=run_id, nested=True):
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("log_loss", logloss)
        mlflow.log_metric("roc_auc", roc_auc)
        
        cm = confusion_matrix(y_test_data, preds)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'{model_name} - Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        cm_file = f"{model_name.replace(' ', '_')}_confusion_matrix.png"
        plt.savefig(cm_file, bbox_inches='tight')
        mlflow.log_artifact(cm_file)
        plt.close()
        os.remove(cm_file)  
        
        plt.figure(figsize=(6, 4))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'{model_name} - ROC Curve')
        plt.legend(loc="lower right")

        roc_file = f"{model_name.replace(' ', '_')}_roc_curve.png"
        plt.savefig(roc_file, bbox_inches='tight')
        mlflow.log_artifact(roc_file)
        plt.close()
        os.remove(roc_file)  

        print(f"\n--- {model_name} Evaluation ---")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Log Loss: {logloss:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")
        print("Classification Report:\n", classification_report(y_test_data, preds))

    return accuracy, roc_auc, logloss

In [9]:
X_train = X_train_scaled
X_test = X_test_scaled

In [10]:
print("\nApproach 1: Filter Methods + Logistic Regression")
with mlflow.start_run(run_name="Filters_LogReg"):
    run_id = mlflow.active_run().info.run_id
    pipeline_approach1 = sklearn.pipeline.Pipeline([
        ('scaler_init', StandardScaler()),
        ('variance_threshold', VarianceThreshold(threshold=0.01)),
        ('kbest', SelectKBest(score_func=f_classif, k=50)),
        ('scaler_final', StandardScaler()),
        ('logreg', LogisticRegression(solver='liblinear', random_state=42, C=0.1, penalty='l1'))
    ])
    mlflow.log_param("approach", "Filters_LogReg")
    mlflow.log_param("variance_threshold", 0.01)
    mlflow.log_param("kbest_k", 50)
    mlflow.log_param("logreg_C", 0.1)
    mlflow.log_param("logreg_penalty", "l1")

    pipeline_approach1.fit(X_train, y_train)
    mlflow.sklearn.log_model(pipeline_approach1, "model")
    print("Approach 1 - Training complete.")
    acc1, roc_auc1, logloss1 = evaluate_model(pipeline_approach1, X_test, y_test, "Filters_LogReg", run_id)


Approach 1: Filter Methods + Logistic Regression
Approach 1 - Training complete.

--- Filters_LogReg Evaluation ---
Accuracy: 0.6000
Log Loss: 0.6405
ROC AUC: 0.6839
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.83      0.71        58
           1       0.55      0.29      0.38        42

    accuracy                           0.60       100
   macro avg       0.58      0.56      0.54       100
weighted avg       0.59      0.60      0.57       100



In [ ]:
# PCA + SVM
import sklearn.decomposition


print("\nApproach 2: PCA + SVM")
with mlflow.start_run(run_name="PCA_SVM"):
    run_id = mlflow.active_run().info.run_id
    scaler_pca = StandardScaler()
    X_train_scaled_pca = scaler_pca.fit_transform(X_train)
    pca_temp = sklearn.decomposition.PCA(n_components=0.95, random_state=42)
    pca_temp.fit(X_train_scaled_pca)
    n_components_pca = pca_temp.n_components_
    print(f"PCA will use {n_components_pca} components to explain 95% variance.")

    mlflow.log_param("approach", "PCA_SVM")
    mlflow.log_param("pca_n_components", n_components_pca)
    mlflow.log_param("svm_C", 0.1)
    mlflow.log_param("svm_kernel", "linear")

    if n_components_pca > 0:
        # PCA variance plot
        plt.figure(figsize=(8, 5))
        plt.plot(np.cumsum(sklearn.decomposition.PCA(n_components=min(X_train.shape[0], X_train.shape[1]), random_state=42).fit(X_train_scaled_pca).explained_variance_ratio_))
        plt.xlabel('Number of Components')
        plt.ylabel('Cumulative Explained Variance')
        plt.title('PCA Explained Variance')
        plt.axhline(y=0.95, color='r', linestyle='--')
        plt.axvline(x=n_components_pca, color='g', linestyle='--')
        plt.grid(True)
        pca_plot_file = "pca_explained_variance.png"
        plt.savefig(pca_plot_file, bbox_inches='tight')
        mlflow.log_artifact(pca_plot_file)
        plt.close()
        os.remove(pca_plot_file)

        pipeline_approach2 = sklearn.pipeline.Pipeline([
            ('scaler', StandardScaler()),
            ('pca', sklearn.decomposition.PCA(n_components=n_components_pca, random_state=42)),
            ('svm', sklearn.svm.SVC(kernel='linear', probability=True, random_state=42, C=0.1))
        ])
        pipeline_approach2.fit(X_train, y_train)
        mlflow.sklearn.log_model(pipeline_approach2, "model")
        print("Approach 2 - Training complete.")
        acc2, roc_auc2, logloss2 = evaluate_model(pipeline_approach2, X_test, y_test, "PCA_SVM", run_id)
    else:
        print("Skipping Approach 2 due to PCA component issue.")
        acc2, roc_auc2, logloss2 = 0, 0, float('inf')
        mlflow.log_metric("accuracy", 0)
        mlflow.log_metric("log_loss", float('inf'))
        mlflow.log_metric("roc_auc", 0)


Approach 2: PCA + SVM
PCA will use 50 components to explain 95% variance.


d:\After\torch\Lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\After\torch\Lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Approach 2 - Training complete.

--- PCA_SVM Evaluation ---
Accuracy: 0.6700
Log Loss: 0.6790
ROC AUC: 0.6905
Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.83      0.74        58
           1       0.66      0.45      0.54        42

    accuracy                           0.67       100
   macro avg       0.67      0.64      0.64       100
weighted avg       0.67      0.67      0.66       100

